In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
import lightgbm as lgb

In [18]:
# ======================
# 1. Load Data 
# ======================

train_data = pd.read_csv('analysis_data.csv')
scoring_data = pd.read_csv('scoring_data.csv')
TARGET_RAW = 'monthly_spend'
TARGET_CLEAN = 'monthly_spend_cleaned'
ID_COL = 'customer_id'

In [19]:
# ======================
# 2. Basic Data Cleaning
# ======================
# Only cap the target variable to avoid data leakage

upper_limit = train_data[TARGET_RAW].quantile(0.99)
train_data[TARGET_CLEAN] = train_data[TARGET_RAW].clip(upper=upper_limit)

In [20]:
# ======================
# 3. Feature Engineering (Sub4 ONLY changes here)
# ======================

def create_features(df):
    """
    Create features for the model
    Sub4: Adds 4 powerful domain-specific features
    """
    # 复制一份，避免修改原始数据
    df = df.copy()
    
    # ----------------------
    # 【基础特征】所有Submission都保留
    # ----------------------
    df['avg_limit_per_card'] = df['credit_limit'] / df['num_credit_cards'].replace(0, 1)
    df['estimated_monthly_volume'] = df['num_transactions'] * df['avg_transaction_value']
    df['monthly_income'] = df['annual_income'] / 12
    df['spend_to_income_ratio'] = df['avg_transaction_value'] / df['monthly_income'].replace(0, 1)
    
    # ----------------------
    # Sub4 NEW FEATURES (Single variable change)
    # ----------------------
    # 1. Credit Utilization: % of credit limit being used (most important feature in credit scoring)
    df['credit_utilization'] = df['avg_transaction_value'] / df['credit_limit'].replace(0, 1)
    
    # 2. Debt-to-Income Ratio: Number of cards relative to annual income
    df['debt_to_income_ratio'] = df['num_credit_cards'] / df['annual_income'].replace(0, 1)
    
    # 3. Card Usage Intensity: Average transactions per credit card
    df['transactions_per_card'] = df['num_transactions'] / df['num_credit_cards'].replace(0, 1)
    
    # 4. Credit Score Tier: Simple binning to capture non-linear effects
    df['credit_score_tier'] = pd.cut(df['credit_score'], bins=5, labels=False)
    
    return df

In [21]:
# ======================
# 4. Define Feature Lists
# ======================

def get_feature_lists(df):
    """
    定义数值特征和类别特征
    """
    # 基础数值特征
    base_num_features = [
        'age', 'annual_income', 'credit_score', 'num_credit_cards', 
        'num_transactions', 'avg_transaction_value', 'credit_limit',
        'avg_limit_per_card', 'estimated_monthly_volume', 'spend_to_income_ratio'
    ]
    
    # 【Sub4-Sub10 核心修改区】添加新的数值特征
    additional_num_features = []  # 后面每次提交会更新这里
    
    num_features = base_num_features + additional_num_features
    
    # 类别特征（固定）
    cat_features = ['education_level', 'gender', 'region', 'employment_status', 'card_type']
    
    return num_features, cat_features

In [22]:
# ======================
# 5. Model Training with 5-Fold Cross Validation
# ======================

def train_and_predict(X, y, X_scoring):
    """
    5折交叉验证训练模型，并返回测试集预测结果
    """
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_rmse = []
    test_preds = np.zeros(len(X_scoring))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # ----------------------
        # 【Sub6-Sub10 核心修改区】替换模型
        # ----------------------
        model = LinearRegression()  # 初始用线性回归，后面替换
        
        model.fit(X_train, y_train)
        
        # 验证集评估
        val_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        fold_rmse.append(rmse)
        print(f'Fold {fold} RMSE: {rmse:.4f}')
        
        # 测试集预测
        test_pred = model.predict(X_scoring)
        test_preds += test_pred / 5
    
    print(f'\nMean CV RMSE: {np.mean(fold_rmse):.4f} (±{np.std(fold_rmse):.4f})')
    return test_preds

In [23]:
# ======================
# 6. Main Pipeline
# ======================

def main():
    # 特征工程
    train_fe = create_features(train_data)
    scoring_fe = create_features(scoring_data)
    
    # 获取特征列
    num_features, cat_features = get_feature_lists(train_fe)
    all_features = num_features + cat_features
    
    # 分离特征和目标
    X = train_fe[all_features]
    y = train_fe[TARGET_CLEAN]
    X_scoring = scoring_fe[all_features]
    
    # 类别特征编码（独热编码，共用）
    X = pd.get_dummies(X, columns=cat_features, drop_first=True)
    X_scoring = pd.get_dummies(X_scoring, columns=cat_features, drop_first=True)
    X, X_scoring = X.align(X_scoring, join='left', axis=1, fill_value=0)
    
    # 缺失值填充（中位数，共用）
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    X_scoring_imputed = pd.DataFrame(imputer.transform(X_scoring), columns=X_scoring.columns)
    
    # 训练模型并预测
    print('='*50)
    print('开始训练...')
    print('='*50)
    predictions = train_and_predict(X_imputed, y, X_scoring_imputed)
    
    # 非负处理（共用）
    predictions = np.maximum(predictions, 0)
    
    # 生成提交文件
    submission = pd.DataFrame({
        ID_COL: scoring_data[ID_COL],
        TARGET_RAW: predictions
    })
    submission.to_csv('submission_v4.csv', index=False)  # 每次改一下版本号
    print('\n提交文件已生成！')

if __name__ == '__main__':
    main()

开始训练...
Fold 1 RMSE: 220.2957
Fold 2 RMSE: 226.4161
Fold 3 RMSE: 226.0194
Fold 4 RMSE: 220.2457
Fold 5 RMSE: 224.4684

Mean CV RMSE: 223.4891 (±2.7073)

提交文件已生成！
